In [1]:
%%writefile kaggle.json
{
  "username": "trishita dhara",
  "key": "KGAT_81a98b8fa6b2bb8b5120dc990bba09cb"
}

Overwriting kaggle.json


In [2]:
!pip -q install kaggle

import os, json

# OPTION A (recommended): upload kaggle.json to Colab (Files sidebar)
# then run this cell.

KAGGLE_JSON_PATH = "/work/kaggle.json"  # change if uploaded elsewhere
assert os.path.exists(KAGGLE_JSON_PATH), "Upload kaggle.json to /content first."

os.makedirs("/root/.kaggle", exist_ok=True)
!cp {KAGGLE_JSON_PATH} /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle credentials set.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Kaggle credentials set.


In [3]:
import os, glob, zipfile

DATA_DIR = "/content/artifact_dataset"
os.makedirs(DATA_DIR, exist_ok=True)

# # # Kaggle dataset slug from your link:
# # # https://www.kaggle.com/datasets/awsaf49/artifact-dataset
!kaggle datasets download -d awsaf49/artifact-dataset -p {DATA_DIR} --force

# # # # Unzip everything in DATA_DIR
zips = glob.glob(os.path.join(DATA_DIR, "*.zip"))
print("Found zips:", zips)

for z in zips:
   with zipfile.ZipFile(z, 'r') as zip_ref:
      zip_ref.extractall(DATA_DIR)
print("Unzipped to:", DATA_DIR)

# Show top-level contents
for p in sorted(os.listdir(DATA_DIR))[:50]:
    print(p)

Dataset URL: https://www.kaggle.com/datasets/awsaf49/artifact-dataset
License(s): other
100%|███████████████████████████████████████| 29.4G/29.4G [03:51<00:00, 136MB/s]

Found zips: ['/content/artifact_dataset/artifact-dataset.zip']
Unzipped to: /content/artifact_dataset
afhq
artifact-dataset.zip
big_gan
celebahq
cips
coco
cycle_gan
ddpm
denoising_diffusion_gan
diffusion_gan
face_synthetics
ffhq
gansformer
gau_gan
generative_inpainting
glide
imagenet
lama
landscape
latent_diffusion
lsun
mat
metfaces
palette
pro_gan
projected_gan
sfhq
stable_diffusion
star_gan
stylegan1
stylegan2
stylegan3
taming_transformer
vq_diffusion


In [4]:
import os, glob, random
import pandas as pd

DATA_DIR = "/content/artifact_dataset"
random.seed(0)

IMG_EXTS = ("*.png","*.jpg","*.jpeg","*.webp")

# Likely "real" sources in this Kaggle dataset
REAL_SOURCES = {
    "ffhq", "celebahq", "afhq", "imagenet", "coco", "lsun", "metfaces", "landscape"
}

# Everything else we treat as "fake generator folders" (but we will explicitly exclude the zip)
IGNORE = {"artifact-dataset.zip"}

def list_images(folder):
    paths = []
    for ext in IMG_EXTS:
        paths.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))
    return sorted(paths)

rows = []
top_folders = [f for f in sorted(os.listdir(DATA_DIR)) if os.path.isdir(os.path.join(DATA_DIR, f)) and f not in IGNORE]

for src in top_folders:
    src_dir = os.path.join(DATA_DIR, src)
    paths = list_images(src_dir)
    if len(paths) == 0:
        continue
    y = 0 if src in REAL_SOURCES else 1
    for p in paths:
        rows.append({"path": p, "y": y, "source": src})

df = pd.DataFrame(rows)
print("Total indexed images:", len(df))
print(df.groupby(["y","source"]).size().sort_values(ascending=False).head(30))
print("\nCounts:", df["y"].value_counts().to_dict())
df.head()

Total indexed images: 2496738
y  source                 
1  stylegan2                  1000000
0  lsun                        539163
   coco                        163846
1  taming_transformer          105000
   stylegan3                    97494
0  imagenet                     96788
   ffhq                         70000
1  mat                          60000
   pro_gan                      40000
0  afhq                         31933
   celebahq                     30000
1  lama                         24705
   generative_inpainting        22000
   stable_diffusion             21444
   glide                        20903
   latent_diffusion             20000
   diffusion_gan                15507
   cycle_gan                    15210
   projected_gan                12000
   cips                         11200
   sfhq                         10000
   denoising_diffusion_gan      10000
   big_gan                      10000
   face_synthetics              10000
   gansformer                  

,path,y,source
0,/content/artifact_dataset/afhq/afhq/afhq/afhq/...,0,afhq
1,/content/artifact_dataset/afhq/afhq/afhq/afhq/...,0,afhq
2,/content/artifact_dataset/afhq/afhq/afhq/afhq/...,0,afhq
3,/content/artifact_dataset/afhq/afhq/afhq/afhq/...,0,afhq
4,/content/artifact_dataset/afhq/afhq/afhq/afhq/...,0,afhq


In [5]:
import numpy as np

# Choose a diverse fake set (GAN + diffusion) to claim cross-family generalization
FAKE_SOURCES = [
    # GAN family
    "stylegan2", "stylegan3", "pro_gan", "cycle_gan", "big_gan",
    # Diffusion family
    "stable_diffusion", "ddpm", "glide", "latent_diffusion", "vq_diffusion"
]

# Real sources subset
REAL_PICK = ["ffhq", "celebahq", "imagenet", "coco", "afhq"]

# How many images per source (keep small first; you can scale later)
N_PER_SOURCE = 200   # => Real: 5*200=1000, Fake: 10*200=2000 (you can lower to 100 if time)

def sample_per_source(df, sources, y, n_per):
    out = []
    for s in sources:
        d = df[(df["source"] == s) & (df["y"] == y)]
        if len(d) == 0:
            print("WARNING missing source:", s)
            continue
        take = min(n_per, len(d))
        out.append(d.sample(take, random_state=0))
    return pd.concat(out, ignore_index=True) if len(out) else pd.DataFrame(columns=df.columns)

real_df = sample_per_source(df, REAL_PICK, y=0, n_per=N_PER_SOURCE)
fake_df = sample_per_source(df, FAKE_SOURCES, y=1, n_per=N_PER_SOURCE)

# Balance final counts: match real and fake totals (optional)
n_real = len(real_df)
n_fake = len(fake_df)
n = min(n_real, n_fake)

real_df = real_df.sample(n, random_state=0)
fake_df = fake_df.sample(n, random_state=0)

eval_df = pd.concat([real_df, fake_df], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

print("Final eval size:", len(eval_df))
print("Real:", (eval_df["y"]==0).sum(), "Fake:", (eval_df["y"]==1).sum())
print("Per-source counts (top):")
print(eval_df.groupby(["y","source"]).size().sort_values(ascending=False).head(20))

# Save for reproducibility
eval_index_path = "/content/artifact_eval_index.csv"
eval_df.to_csv(eval_index_path, index=False)
print("Saved:", eval_index_path)

Final eval size: 2000
Real: 1000 Fake: 1000
Per-source counts (top):
y  source          
0  afhq                200
   celebahq            200
   coco                200
   ffhq                200
   imagenet            200
1  pro_gan             116
   vq_diffusion        105
   big_gan             102
   glide               101
   ddpm                101
   stable_diffusion     98
   stylegan2            96
   stylegan3            95
   cycle_gan            94
   latent_diffusion     92
dtype: int64
Saved: /content/artifact_eval_index.csv


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as tvt

IMG_SIZE = 224
BATCH = 32

transform = tvt.Compose([
    tvt.Resize((IMG_SIZE, IMG_SIZE)),
    tvt.ToTensor(),  # [0,1]
])

class ImageDFDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        x = self.transform(img)
        return x, int(row["y"]), row["source"], row["path"]

eval_ds = ImageDFDataset(eval_df, transform)
eval_loader = DataLoader(eval_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Loader ready:", len(eval_ds))

Loader ready: 2000


In [7]:
import os, torch, torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

CLIP_CKPT_PATH   = "/work/paper_outputs/clip_detector.pt"
RESNET_CKPT_PATH = "/work/paper_outputs/resnet_detector.pt"
DELTA_PATH       = "/work/paper_outputs/ensemble_delta_bottom_eps16.pt"  # optional

assert os.path.exists(CLIP_CKPT_PATH), CLIP_CKPT_PATH
assert os.path.exists(RESNET_CKPT_PATH), RESNET_CKPT_PATH
print("✅ checkpoints found")

device: cpu
✅ checkpoints found


In [8]:
def load_state_dict_flexible(path):
    obj = torch.load(path, map_location=device)
    if isinstance(obj, dict):
        for k in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
        # already a state_dict
        if all(isinstance(k, str) for k in obj.keys()):
            return obj
    raise ValueError(f"Unrecognized checkpoint format for {path}. Type={type(obj)}, keys={list(obj.keys())[:20] if isinstance(obj, dict) else None}")

In [15]:
!pip -q install open_clip_torch
import torch, torch.nn as nn
import open_clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths you provided
CLIP_CKPT_PATH = "/work/paper_outputs/clip_detector.pt"

def load_state_dict_flexible(path):
    obj = torch.load(path, map_location=device)
    if isinstance(obj, dict):
        for k in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
        return obj
    raise ValueError(f"Unrecognized checkpoint format for {path}")

sd = load_state_dict_flexible(CLIP_CKPT_PATH)
sd = {k.replace("module.", ""): v for k, v in sd.items()}

# find head weight key
head_w_key = None
for k in ["head.weight", "classifier.weight", "fc.weight"]:
    if k in sd:
        head_w_key = k
        break
assert head_w_key is not None, f"Can't find head weight key. Sample keys: {list(sd.keys())[:30]}"

ckpt_dim = int(sd[head_w_key].shape[1])
print("Checkpoint head input dim:", ckpt_dim)

# map dim -> CLIP backbone
DIM_TO_BACKBONE = {
    512: ("ViT-B-32", "openai"),
    768: ("ViT-L-14", "openai"),
    1024: ("ViT-H-14", "laion2b_s32b_b79k"),  # if you ever hit this, tell me; pretrained may differ
}

assert ckpt_dim in DIM_TO_BACKBONE, f"Unexpected ckpt_dim={ckpt_dim}. Tell me ckpt_dim and the checkpoint keys."
CLIP_MODEL_NAME, CLIP_PRETRAINED = DIM_TO_BACKBONE[ckpt_dim]
print("Using backbone:", CLIP_MODEL_NAME, "| pretrained:", CLIP_PRETRAINED)

clip_backbone, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
clip_backbone = clip_backbone.to(device).eval()
for p in clip_backbone.parameters():
    p.requires_grad = False

feat_dim = clip_backbone.visual.output_dim
print("Backbone output_dim:", feat_dim)

class CLIPDetector(nn.Module):
    def __init__(self, clip_model, feat_dim):
        super().__init__()
        self.clip = clip_model
        self.head = nn.Linear(feat_dim, 2)
    def forward(self, x):
        feats = self.clip.encode_image(x)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        return self.head(feats)

clip_detector = CLIPDetector(clip_backbone, feat_dim).to(device)

missing, unexpected = clip_detector.load_state_dict(sd, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

clip_detector.eval()
print("✅ CLIP detector loaded successfully")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Checkpoint head input dim: 768
Using backbone: ViT-L-14 | pretrained: openai
/root/venv/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Backbone output_dim: 768
Missing keys: ['clip.positional_embedding', 'clip.text_projection', 'clip.logit_scale', 'clip.visual.class_embedding', 'clip.visual.positional_embedding', 'clip.visual.proj', 'clip.visual.conv1.weight', 'clip.visual.ln_pre.weight', 'clip.visual.ln_pre.bias', 'clip.visual.transformer.resblocks.0.ln_1.weight', 'clip.visual.transformer.resblocks.0.ln_1.bias', 'clip.visual.transformer.resblocks.0.attn.in_proj_weight', 'clip.visual.transformer.resblocks.0.attn.in_proj_bias', 'clip.visual.transformer.resblocks.0.attn.out_proj.weight', 'clip.visual.transformer.resblocks.0.attn.ou

In [18]:
resnet_detector = models.resnet18(weights=None)
resnet_detector.fc = nn.Linear(resnet_detector.fc.in_features, 2)
resnet_detector = resnet_detector.to(device)

sd = load_state_dict_flexible(RESNET_CKPT_PATH)
try:
    resnet_detector.load_state_dict(sd, strict=True)
except Exception as e:
    print("Strict load failed for ResNet. Trying prefix fixes...")
    sd2 = {k.replace("module.", ""): v for k, v in sd.items()}
    missing, unexpected = resnet_detector.load_state_dict(sd2, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

resnet_detector.eval()
print("✅ ResNet detector loaded")

Strict load failed for ResNet. Trying prefix fixes...
Missing keys: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.conv2.weight', 'layer1.1.bn2.weight', 'layer1.1.bn2.bias', 'layer1.1.bn2.running_mean', 'layer1.1.bn2.running_var', 'layer2.0.conv1.weight', 'layer2.0.bn1.weight', 'layer2.0.bn1.bias', 'layer2.0.bn1.running_mean', 'layer2.0.bn1.running_var', 'layer2.0.conv2.weight', 'layer2.0.bn2.weight', 'layer2.0.bn2.bias', 'layer2.0.bn2.running_mean', 'layer2.0.bn2.running_var', 'layer2.0.downsample.0.weight', 'layer2.0.downsample.1.weight', 'layer2.0.downsample.1.bias', 

In [21]:
@torch.no_grad()
def p_fake_from_logits(logits):
    if logits.ndim == 2 and logits.shape[1] == 2:
        return torch.softmax(logits, dim=1)[:, 1]
    if logits.ndim == 2 and logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])
    if logits.ndim == 1:
        return logits.clamp(0, 1)
    raise ValueError(f"Unexpected logits shape: {logits.shape}")

class EnsembleDetector(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)
    def forward(self, x):
        ps = []
        for m in self.models:
            m.eval()
            ps.append(p_fake_from_logits(m(x)))
        p_fake = torch.stack(ps, dim=0).mean(dim=0)
        p_real = 1.0 - p_fake
        return torch.stack([p_real, p_fake], dim=1)

ensemble_detector = EnsembleDetector([clip_detector, resnet_detector]).to(device).eval()
print("✅ Ensemble detector ready")

✅ Ensemble detector ready


In [24]:
delta = torch.load(DELTA_PATH, map_location="cpu")
print("Loaded delta:", type(delta), getattr(delta, "shape", None), "max|delta|=", float(delta.abs().max()))

Loaded delta: <class 'torch.Tensor'> torch.Size([1, 3, 224, 224]) max|delta|= 0.062745101749897


In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score

@torch.no_grad()
def eval_clean(model, loader):
    model.eval()
    ys, ps, srcs = [], [], []
    for x, y, src, _ in loader:
        x = x.to(device)
        logits = model(x)
        p_fake = p_fake_from_logits(logits).detach().cpu().numpy()
        ys.append(np.array(y))
        ps.append(p_fake)
        srcs.extend(list(src))
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    overall = {
        "auc": float(roc_auc_score(y, p)),
        "acc@0.5": float(accuracy_score(y, (p >= 0.5).astype(int))),
        "fake_to_real@0.5": float((p[y==1] < 0.5).mean()),
        "n": int(len(y)),
    }
    # Per-source metrics (AUC only meaningful if source slice contains both classes)
    per_rows = []
    srcs = np.array(srcs)
    for s in sorted(set(srcs)):
        idx = (srcs == s)
        ys_s, ps_s = y[idx], p[idx]
        row = {"source": s, "n": int(idx.sum())}
        if len(np.unique(ys_s)) >= 2:
            row["auc"] = float(roc_auc_score(ys_s, ps_s))
        else:
            row["auc"] = np.nan
        row["acc@0.5"] = float(accuracy_score(ys_s, (ps_s >= 0.5).astype(int)))
        row["fake_to_real@0.5"] = float((ps_s[ys_s==1] < 0.5).mean()) if (ys_s==1).any() else np.nan
        per_rows.append(row)
    per_df = pd.DataFrame(per_rows).sort_values(["auc","acc@0.5"], ascending=True)
    return overall, per_df

results = []
per_source_all = []

for name, m in [("CLIP", clip_detector), ("ResNet18", resnet_detector), ("Ensemble", ensemble_detector)]:
    overall, per_df = eval_clean(m, eval_loader)
    overall["model"] = name
    results.append(overall)
    per_df["model"] = name
    per_source_all.append(per_df)
    print(name, overall)

overall_df = pd.DataFrame(results)
per_source_df = pd.concat(per_source_all, ignore_index=True)

overall_df.to_csv("/content/artifact_clean_overall_all_models.csv", index=False)
per_source_df.to_csv("/content/artifact_clean_per_source_all_models.csv", index=False)

print("Saved:",
      "/content/artifact_clean_overall_all_models.csv",
      "/content/artifact_clean_per_source_all_models.csv")
overall_df

/root/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipykernel_74/2521139704.py:13: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  ys.append(np.array(y))
CLIP {'auc': 0.455438, 'acc@0.5': 0.5, 'fake_to_real@0.5': 1.0, 'n': 2000, 'model': 'CLIP'}
/root/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipykernel_74/2521139704.py:13: DeprecationWarning: __array__ implementation doesn't accept a 

,auc,acc@0.5,fake_to_real@0.5,n,model
0,0.455438,0.5000,1.000,2000,CLIP
1,0.559713,0.5000,0.000,2000,ResNet18
2,0.554270,0.4975,0.005,2000,Ensemble


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=dbc8aaad-7458-4d3a-af75-0f37995653fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>